# Pinecone + ThaiLLM Judge v02

Improvements over v01:

1. Chain-of-thought prompting with `<think>` blocks
2. De-biased answer 9/10 instructions
3. Improved answer parsing (last number after think block)
4. TOP_K increased to 10
5. Keyword-based policy injection
6. Product summary table for budget/filter questions
7. Two-pass retry for suspicious answer 9/10
8. System message for expert persona


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from collections import OrderedDict
from datetime import datetime
from pathlib import Path
from typing import Any
import subprocess

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from markdown_it import MarkdownIt
from openai import OpenAI
from pinecone import Pinecone
from tqdm.notebook import tqdm

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 240)


In [2]:
INDEX_NAME = "fahmai-text-embedding-3-large"
NAMESPACE = "knowledge-base-markdown"
EMBEDDING_MODEL = "openai/text-embedding-3-large"
EMBEDDING_DIMENSIONS = 1024
TOP_K = 10
THAILLM_URL = "http://thaillm.or.th/api/typhoon/v1/chat/completions"
THAILLM_MODEL = "/model"
THAILLM_TEMPERATURE = 0.0
THAILLM_MAX_TOKENS = 1024
REQUEST_SLEEP_SECONDS = 0.4
REQUEST_MAX_RETRIES = 3
MAX_QUESTIONS = None
RETRY_ON_9_10_THRESHOLD = 0.45
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d-%H%M")
OUTPUT_SUBMISSION_PATH = Path(f"outputs/{RUN_TIMESTAMP}-submission_thaillm_v02.csv")
OUTPUT_DEBUG_PATH = Path(f"outputs/{RUN_TIMESTAMP}-debug_thaillm_v02.csv")


In [3]:
def load_env_from_parents(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        env_path = base / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


def resolve_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "knowledge_base").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the competition project directory from the current working directory."
    )


env_path = load_env_from_parents()
PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data"
KB_DIR = DATA_DIR / "knowledge_base"
QUESTIONS_PATH = DATA_DIR / "questions.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
thaillm_api_key = os.getenv("THAILLM_API_KEY")
openrouter_site_url = os.getenv("OPENROUTER_SITE_URL")
openrouter_site_title = os.getenv("OPENROUTER_SITE_TITLE", "FahMai ThaiLLM Judge")

missing = [
    name
    for name, value in {
        "OPENROUTER_API_KEY": openrouter_api_key,
        "PINECONE_API_KEY": pinecone_api_key,
        "THAILLM_API_KEY": thaillm_api_key,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

print(f"Loaded .env from: {env_path}" if env_path else "Using shell environment only.")
print(f"Project directory: {PROJECT_DIR}")
print(f"ThaiLLM key preview: {thaillm_api_key[:6]}...{thaillm_api_key[-4:]}")


Loaded .env from: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/.env
Project directory: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1
ThaiLLM key preview: TGauKr...6cTE


In [4]:
embedding_client = OpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key
)
pc = Pinecone(api_key=pinecone_api_key)
index_description = pc.describe_index(INDEX_NAME)
index_info = (
    index_description.to_dict()
    if hasattr(index_description, "to_dict")
    else dict(index_description)
)
index_dimension = index_info.get("dimension") or index_info.get("spec", {}).get(
    "dimension"
)
if index_dimension != EMBEDDING_DIMENSIONS:
    raise RuntimeError(
        f"Index {INDEX_NAME!r} has dimension {index_dimension}, expected {EMBEDDING_DIMENSIONS}."
    )
index = pc.Index(INDEX_NAME)

questions_df = pd.read_csv(QUESTIONS_PATH)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
if MAX_QUESTIONS is not None:
    questions_df = questions_df.head(MAX_QUESTIONS).copy()

display(
    pd.DataFrame(
        [
            {
                "index_name": INDEX_NAME,
                "dimension": index_dimension,
                "questions": len(questions_df),
            }
        ]
    )
)


,index_name,dimension,questions
0,fahmai-text-embedding-3-large,1024,100


In [5]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


THAI_CHAR_RE = re.compile(r"[ก-๙]")
ALNUM_TOKEN_RE = re.compile(r"[a-z0-9][a-z0-9.+/-]*")
NUMBER_TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)?")
THAI_WORDLIKE_RE = re.compile(r"[ก-๙]+")


def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("×", "x")
    text = re.sub(r"[^0-9a-zก-๙.%/+-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def thai_char_ngrams(text: str, n: int = 3) -> list[str]:
    compact = re.sub(r"[^ก-๙]", "", text)
    if len(compact) < n:
        return [compact] if compact else []
    return [compact[i : i + n] for i in range(len(compact) - n + 1)]


def tokenize_for_overlap(text: str) -> list[str]:
    normalized = normalize_text(text)
    ascii_tokens = ALNUM_TOKEN_RE.findall(normalized)
    thai_tokens = THAI_WORDLIKE_RE.findall(normalized)
    thai_ngrams = []
    for token in thai_tokens:
        thai_ngrams.extend(thai_char_ngrams(token, n=3))
    tokens = (
        ascii_tokens + thai_tokens + thai_ngrams + NUMBER_TOKEN_RE.findall(normalized)
    )
    return [token for token in tokens if token]


def extract_support_tokens(text: str) -> set[str]:
    normalized = normalize_text(text)
    parts = set(ALNUM_TOKEN_RE.findall(normalized))
    parts.update(NUMBER_TOKEN_RE.findall(normalized))
    parts.update(
        token for token in THAI_WORDLIKE_RE.findall(normalized) if len(token) >= 2
    )
    return {token for token in parts if token}


def estimate_token_count(text: str) -> int:
    cleaned = normalize_whitespace(text)
    return max(1, math.ceil(len(cleaned) / 4)) if cleaned else 0


def render_heading_lines(path: list[str], start_level: int = 1) -> str:
    if not path:
        return ""
    return "\n".join(
        f"{'#' * (start_level + idx)} {heading}" for idx, heading in enumerate(path)
    )


def dedupe_block_texts(block_texts: list[str]) -> list[str]:
    deduped: list[str] = []
    for text in block_texts:
        normalized = normalize_whitespace(text)
        if not normalized:
            continue
        if any(
            normalized == existing or normalized in existing for existing in deduped
        ):
            continue
        deduped = [existing for existing in deduped if existing not in normalized]
        deduped.append(normalized)
    return deduped


def compose_section_text(heading_path: list[str], block_texts: list[str]) -> str:
    pieces = []
    heading_lines = render_heading_lines(heading_path)
    if heading_lines:
        pieces.append(heading_lines)
    pieces.extend(dedupe_block_texts(block_texts))
    return normalize_whitespace("\n\n".join(pieces))


def parse_markdown_blocks(text: str, source_path: str) -> list[dict[str, Any]]:
    md = MarkdownIt("commonmark").enable("table").enable("strikethrough")
    tokens = md.parse(text)
    lines = text.splitlines()
    stack: list[str | None] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.type == "heading_open":
            level = int(token.tag[1])
            inline_token = tokens[i + 1] if i + 1 < len(tokens) else None
            heading_text = normalize_whitespace(getattr(inline_token, "content", ""))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            i += 1
            continue
        if token.type in {
            "paragraph_open",
            "bullet_list_open",
            "ordered_list_open",
            "table_open",
        }:
            start_line, end_line = token.map or (None, None)
            segment = (
                "\n".join(lines[start_line:end_line])
                if start_line is not None and end_line is not None
                else ""
            )
            segment = normalize_whitespace(segment)
            if segment:
                blocks.append(
                    {
                        "source_path": source_path,
                        "block_type": token.type.removesuffix("_open"),
                        "heading_path": current_path(),
                        "text": segment,
                    }
                )
        elif token.type == "fence" and token.content:
            fence_text = normalize_whitespace(token.markup + "\n" + token.content)
            blocks.append(
                {
                    "source_path": source_path,
                    "block_type": "fence",
                    "heading_path": current_path(),
                    "text": fence_text,
                }
            )
        i += 1
    return blocks


def build_sections(
    blocks: list[dict[str, Any]], merge_preamble_tokens: int = 80
) -> list[dict[str, Any]]:
    grouped: OrderedDict[tuple[str, ...], list[dict[str, Any]]] = OrderedDict()
    for block in blocks:
        key = tuple(block["heading_path"])
        grouped.setdefault(key, []).append(block)
    sections: list[dict[str, Any]] = []
    for key, grouped_blocks in grouped.items():
        block_texts = [block["text"] for block in grouped_blocks]
        section_text = compose_section_text(list(key), block_texts)
        sections.append(
            {
                "source_path": grouped_blocks[0]["source_path"],
                "heading_path": list(key),
                "block_count": len(grouped_blocks),
                "text": section_text,
                "estimated_tokens": estimate_token_count(section_text),
            }
        )
    if len(sections) >= 2:
        first_section = sections[0]
        second_section = sections[1]
        if (
            len(first_section["heading_path"]) == 1
            and len(second_section["heading_path"]) >= 2
            and second_section["heading_path"][0] == first_section["heading_path"][0]
            and first_section["estimated_tokens"] <= merge_preamble_tokens
        ):
            merged_parts = [first_section["text"]]
            child_heading_lines = render_heading_lines(
                second_section["heading_path"][1:], start_level=2
            )
            if child_heading_lines:
                merged_parts.append(child_heading_lines)
            merged_parts.append(
                "\n\n".join(
                    block["text"]
                    for block in grouped[tuple(second_section["heading_path"])]
                )
            )
            sections[1] = {
                **second_section,
                "block_count": first_section["block_count"]
                + second_section["block_count"],
                "text": normalize_whitespace(
                    "\n\n".join(part for part in merged_parts if part)
                ),
            }
            sections[1]["estimated_tokens"] = estimate_token_count(sections[1]["text"])
            sections = sections[1:]
    return sections


def split_text_with_overlap(
    text: str, chunk_size: int = 450, chunk_overlap: int = 75
) -> list[str]:
    text = normalize_whitespace(text)
    if estimate_token_count(text) <= chunk_size:
        return [text]
    separators = ["\n\n", "\n", ". ", " "]
    pieces = [text]
    for separator in separators:
        if all(estimate_token_count(piece) <= chunk_size for piece in pieces):
            break
        next_pieces: list[str] = []
        for piece in pieces:
            if estimate_token_count(piece) <= chunk_size or separator not in piece:
                next_pieces.append(piece)
                continue
            split_parts = [
                normalize_whitespace(part)
                for part in piece.split(separator)
                if normalize_whitespace(part)
            ]
            if separator in {". ", " "}:
                suffix = "." if separator == ". " else ""
                split_parts = [
                    part + suffix if suffix and not part.endswith(suffix) else part
                    for part in split_parts
                ]
            next_pieces.extend(split_parts)
        pieces = next_pieces
    hard_limit = max(200, chunk_size * 4)
    flattened: list[str] = []
    for piece in pieces:
        if estimate_token_count(piece) <= chunk_size:
            flattened.append(piece)
            continue
        for start in range(0, len(piece), hard_limit):
            flattened.append(piece[start : start + hard_limit])
    chunks: list[str] = []
    current = ""
    overlap_chars = max(80, chunk_overlap * 6)
    for piece in flattened:
        piece = normalize_whitespace(piece)
        if not piece:
            continue
        candidate = normalize_whitespace(f"{current}\n\n{piece}" if current else piece)
        if current and estimate_token_count(candidate) > chunk_size:
            chunks.append(current)
            overlap_seed = current[-overlap_chars:]
            current = normalize_whitespace(f"{overlap_seed}\n\n{piece}")
            if estimate_token_count(current) > chunk_size:
                chunks.append(piece)
                current = ""
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def build_rag_records(
    sections: list[dict[str, Any]], chunk_size: int = 450, chunk_overlap: int = 75
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    next_chunk_number = 0
    for section_index, section in enumerate(sections):
        child_chunks = split_text_with_overlap(
            section["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        for chunk_in_section, chunk_text in enumerate(child_chunks):
            chunk_id = f"{section['source_path']}#chunk_{next_chunk_number:04d}"
            content_hash = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
            records.append(
                {
                    "id": chunk_id,
                    "chunk_text": chunk_text,
                    "chunk_tokens": estimate_token_count(chunk_text),
                    "metadata": {
                        "content_hash": content_hash,
                        "document_id": section["source_path"],
                        "file_path": section["source_path"],
                        "heading_path": section["heading_path"],
                        "section_index": section_index,
                        "chunk_number": next_chunk_number,
                        "chunk_in_section": chunk_in_section,
                        "source_type": "markdown",
                        "text_preview": chunk_text[:200],
                    },
                }
            )
            next_chunk_number += 1
    return records


def load_markdown_corpus(root: Path) -> list[dict[str, Any]]:
    records = []
    for path in sorted(root.rglob("*.md")):
        records.append(
            {
                "source_path": str(path.relative_to(root.parent)),
                "path": path,
                "text": path.read_text(encoding="utf-8"),
            }
        )
    return records


def build_all_chunks(root: Path) -> list[dict[str, Any]]:
    all_records: list[dict[str, Any]] = []
    for doc in tqdm(load_markdown_corpus(root), desc="Build local chunks"):
        blocks = parse_markdown_blocks(doc["text"], doc["source_path"])
        sections = build_sections(blocks)
        all_records.extend(build_rag_records(sections))
    return all_records


In [6]:
chunk_records = build_all_chunks(KB_DIR)
chunk_records_by_id = {record["id"]: record for record in chunk_records}

DOMAIN_TERMS = set()
for record in chunk_records:
    DOMAIN_TERMS.update(
        token
        for token in extract_support_tokens(record["metadata"]["file_path"])
        if len(token) >= 2
    )
for question in questions_df["question"].tolist():
    DOMAIN_TERMS.update(
        token for token in extract_support_tokens(str(question)) if len(token) >= 2
    )

OUT_OF_DOMAIN_HINTS = {
    "การเมือง",
    "ฟุตบอล",
    "ดารา",
    "หวย",
    "โรงแรม",
    "เที่ยวบิน",
    "ภาษีที่ดิน",
    "สูตรอาหาร",
    "สุขภาพ",
    "การลงทุน",
    "จังหวัดไหน",
    "นายก",
}
QUESTION_WEAK_SCORE = 0.35
OUT_OF_DOMAIN_LENGTH = 90

# --- Policy keyword injection mapping ---
POLICY_KEYWORD_MAP = {
    "membership_points_policy.md": [
        "points",
        "fahmai points",
        "สมาชิก",
        "gold",
        "silver",
        "platinum",
        "คะแนน",
        "พอยท์",
        "แต้ม",
    ],
    "return_policy.md": ["คืนสินค้า", "คืน", "เงินคืน", "return", "ส่งคืน"],
    "shipping_policy.md": ["จัดส่ง", "ค่าส่ง", "shipping", "ขนส่ง", "ค่าจัดส่ง"],
    "warranty_policy.md": ["ประกัน", "เคลม", "warranty", "care+", "ซ่อม"],
    "cancellation_policy.md": ["ยกเลิก", "cancel", "ยกเลิกคำสั่งซื้อ"],
}

# Build lookup: policy file -> list of chunk records
policy_chunks_by_file: dict[str, list[dict]] = {}
for record in chunk_records:
    fp = record["metadata"]["file_path"]
    if "policies/" in fp:
        fname = fp.split("/")[-1]
        policy_chunks_by_file.setdefault(fname, []).append(record)


def get_policy_chunks_for_question(question_text: str) -> list[dict]:
    """Return policy chunks that match keywords in the question."""
    normalized = normalize_text(question_text)
    injected = []
    for policy_file, keywords in POLICY_KEYWORD_MAP.items():
        if any(kw in normalized for kw in keywords):
            injected.extend(policy_chunks_by_file.get(policy_file, []))
    return injected


# --- Product summary table for budget/filter questions ---
def build_product_summary() -> list[dict]:
    summaries = []
    for path in sorted((KB_DIR / "products").glob("*.md")):
        text = path.read_text(encoding="utf-8")
        lines = text.strip().split("\n")
        name = lines[0].replace("# ", "").strip() if lines else path.stem
        price = None
        category = None
        status = None
        product_code = None
        for line in lines[:15]:
            line = line.strip()
            if line.startswith("ราคา:"):
                price_match = re.search(r"[\d,]+", line.replace("฿", ""))
                if price_match:
                    price = int(price_match.group().replace(",", ""))
            elif line.startswith("หมวดหมู่:"):
                category = line.split(":", 1)[1].strip()
            elif line.startswith("สถานะ:"):
                status = line.split(":", 1)[1].strip()
            elif line.startswith("รหัสสินค้า:"):
                product_code = line.split(":", 1)[1].strip()
        summaries.append(
            {
                "name": name,
                "code": product_code,
                "category": category,
                "price": price,
                "status": status,
                "file": str(path.relative_to(KB_DIR.parent)),
            }
        )
    return summaries


product_summaries = build_product_summary()
BUDGET_PATTERN = re.compile(
    r"(?:งบ|เงิน|ราคาไม่เกิน|ไม่เกิน|ต่ำกว่า|budget)[\s฿]*([\d,]+)", re.IGNORECASE
)
CATEGORY_KEYWORDS = {
    "ลำโพง": "ลำโพง",
    "หูฟัง": "หูฟัง",
    "สมาร์ทวอทช์": "สมาร์ทวอทช์",
    "นาฬิกา": "สมาร์ทวอทช์",
    "แท็บเล็ต": "แท็บเล็ต",
    "แล็ปท็อป": "แล็ปท็อป",
    "โน้ตบุ๊ก": "แล็ปท็อป",
    "มือถือ": "สมาร์ทโฟน",
    "สมาร์ทโฟน": "สมาร์ทโฟน",
    "TWS": "หูฟัง",
    "earbuds": "หูฟัง",
    "tws": "หูฟัง",
}


def get_budget_context(question_text: str) -> str | None:
    """For budget/filter questions, return a summary of matching products."""
    budget_match = BUDGET_PATTERN.search(question_text)
    if not budget_match:
        return None
    budget = int(budget_match.group(1).replace(",", ""))
    # Detect category from question
    detected_cats = set()
    normalized_q = normalize_text(question_text)
    for keyword, cat in CATEGORY_KEYWORDS.items():
        if keyword.lower() in normalized_q or keyword in question_text:
            detected_cats.add(cat)
    matching = []
    for p in product_summaries:
        if p["price"] is None:
            continue
        if p["price"] > budget:
            continue
        if (
            detected_cats
            and p["category"]
            and not any(cat in p["category"] for cat in detected_cats)
        ):
            continue
        matching.append(p)
    if not matching:
        return None
    lines = [f"สินค้าที่ราคาไม่เกิน ฿{budget:,}:"]
    for p in sorted(matching, key=lambda x: x["price"]):
        lines.append(
            f"- {p['name']} ({p['code']}) | หมวด: {p['category']} | ราคา: ฿{p['price']:,}"
        )
    return "\n".join(lines)


print(f"Local chunk records: {len(chunk_records)}")
print(f"Product summaries: {len(product_summaries)}")
print(f"Policy chunk groups: {list(policy_chunks_by_file.keys())}")
display(
    pd.DataFrame(
        {
            "id": [record["id"] for record in chunk_records[:10]],
            "heading_path": [
                " > ".join(record["metadata"]["heading_path"])
                for record in chunk_records[:10]
            ],
            "preview": [record["chunk_text"][:160] for record in chunk_records[:10]],
        }
    )
)


Build local chunks:   0%|          | 0/118 [00:00<?, ?it/s]

Local chunk records: 753
Product summaries: 110
Policy chunk groups: ['cancellation_policy.md', 'membership_points_policy.md', 'return_policy.md', 'shipping_policy.md', 'warranty_policy.md']


,id,heading_path,preview
0,knowledge_base/policies/cancellation_policy.md#chunk_0000,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 1. ภาพรวมนโยบาย,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n\n**วันที่อัปเดต:** 1 มีนาคม 2569\n\n## 1. ภาพรวมนโยบาย\n\nฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุ
1,knowledge_base/policies/cancellation_policy.md#chunk_0001,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)","# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)\n\n**ยกเลิกได้ทันที**\n\nคำสั่งซื้อที่อยู่ใ"
2,knowledge_base/policies/cancellation_policy.md#chunk_0002,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.2 สถานะ ""ชำระเงินแล้ว — กำลังเตรียมจัดส่ง"" (Processing / Preparing to Ship)","# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.2 สถานะ ""ชำระเงินแล้ว — กำลังเตรียมจัดส่ง"" (Processing / Preparing to Ship)\n\n*"
3,knowledge_base/policies/cancellation_policy.md#chunk_0003,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.3 สถานะ ""จัดส่งแล้ว"" (Shipped / In Transit)","# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.3 สถานะ ""จัดส่งแล้ว"" (Shipped / In Transit)\n\n**ไม่สามารถยกเลิกได้**\n\nเมื่อสินค"
4,knowledge_base/policies/cancellation_policy.md#chunk_0004,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n\nสินค้า Pre-order (สั่งจองล่วงหน้า) มีเงื่อนไขการยกเลิกที่แตกต่างจากสินค้าทั่วไป
5,knowledge_base/policies/cancellation_policy.md#chunk_0005,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย**,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย**\n\nหาก
6,knowledge_base/policies/cancellation_policy.md#chunk_0006,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**\n\nหากยกเลิกเม
7,knowledge_base/policies/cancellation_policy.md#chunk_0007,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.3 หลังวันจัดส่ง — ถือเป็นการคืนสินค้า,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.3 หลังวันจัดส่ง — ถือเป็นการคืนสินค้า\n\nหากคำสั่งซื้อ Pre-order ผ่านวันจัดส
8,knowledge_base/policies/cancellation_policy.md#chunk_0008,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation),# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation)\n\nในบางกรณี ฟ้าใหม่อาจต้องยกเลิกคำสั่งซื้อของลูกค้าโดยไม่ได้ร
9,knowledge_base/policies/cancellation_policy.md#chunk_0009,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation) > 4.1 เหตุผลที่ฟ้าใหม่อาจยกเลิกคำสั่งซื้อ,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation)\n### 4.1 เหตุผลที่ฟ้าใหม่อาจยกเลิกคำสั่งซื้อ\n\n| สาเหตุ | รายล


In [7]:
def embed_text_batch(texts: list[str]) -> list[list[float]]:
    headers = {"X-OpenRouter-Title": openrouter_site_title}
    if openrouter_site_url:
        headers["HTTP-Referer"] = openrouter_site_url
    response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
        encoding_format="float",
        dimensions=EMBEDDING_DIMENSIONS,
        extra_headers=headers,
    )
    vectors = [item.embedding for item in response.data]
    if len(vectors) != len(texts):
        raise RuntimeError(
            f"Expected {len(texts)} embeddings, got {len(vectors)} from {EMBEDDING_MODEL}."
        )
    for idx, vector in enumerate(vectors):
        if len(vector) != EMBEDDING_DIMENSIONS:
            raise RuntimeError(
                f"Embedding dimension mismatch at item {idx}: expected {EMBEDDING_DIMENSIONS}, got {len(vector)}."
            )
    return vectors


def retrieve_chunks(question_text: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    vector = embed_text_batch([question_text])[0]
    response = index.query(
        vector=vector, top_k=top_k, namespace=NAMESPACE, include_metadata=True
    )
    rows: list[dict[str, Any]] = []
    seen_ids: set[str] = set()
    for match in getattr(response, "matches", []):
        payload = match.to_dict() if hasattr(match, "to_dict") else match
        record_id = payload.get("id")
        if record_id in seen_ids:
            continue
        seen_ids.add(record_id)
        local_record = chunk_records_by_id.get(record_id, {})
        metadata = payload.get("metadata", {})
        rows.append(
            {
                "id": record_id,
                "score": float(payload.get("score", 0.0)),
                "file_path": metadata.get("file_path"),
                "heading_path": metadata.get("heading_path", []),
                "chunk_text": local_record.get(
                    "chunk_text", metadata.get("text_preview", "")
                ),
                "text_preview": metadata.get("text_preview", ""),
            }
        )
    # Inject policy chunks based on question keywords
    policy_extras = get_policy_chunks_for_question(question_text)
    for record in policy_extras:
        rid = record["id"]
        if rid not in seen_ids:
            seen_ids.add(rid)
            rows.append(
                {
                    "id": rid,
                    "score": 0.40,
                    "file_path": record["metadata"]["file_path"],
                    "heading_path": record["metadata"]["heading_path"],
                    "chunk_text": record["chunk_text"],
                    "text_preview": record["chunk_text"][:200],
                }
            )
    return rows


In [8]:
def strip_think_blocks(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def extract_answer_id_from_text(text: str) -> int:
    """Extract answer ID, preferring the last standalone number after think blocks."""
    cleaned = strip_think_blocks(text)
    # Try last standalone number on its own line
    lines = [line.strip() for line in cleaned.strip().split("\n") if line.strip()]
    for line in reversed(lines):
        m = re.fullmatch(r"(10|[1-9])", line)
        if m:
            return int(m.group(1))
    # Try first standalone number
    exact_match = re.fullmatch(r"\s*(10|[1-9])\s*", cleaned)
    if exact_match:
        return int(exact_match.group(1))
    line_start_match = re.match(r"^\s*(10|[1-9])\b", cleaned)
    if line_start_match:
        return int(line_start_match.group(1))
    patterns = [
        r'"answer_id"\s*:\s*"?(10|[1-9])"?',
        r"\banswer_id\b\s*(?:is|=|:)\s*(10|[1-9])\b",
        r"\b(?:option|choice|answer)\s*(10|[1-9])\b",
        r"คำตอบ(?:คือ)?\s*(10|[1-9])\b",
        r"ตอบ\s*(10|[1-9])\b",
        r"ตัวเลือก\s*(10|[1-9])\b",
        r"ข้อ\s*(10|[1-9])\b",
    ]
    for pattern in patterns:
        match = re.search(pattern, cleaned, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    numbers = re.findall(r"\b(10|[1-9])\b", cleaned)
    if len(numbers) == 1:
        return int(numbers[0])
    # If multiple numbers found, take the last one (likely the final answer)
    if numbers:
        return int(numbers[-1])
    raise ValueError("Could not extract answer_id from ThaiLLM text output")


def call_thaillm(
    messages: list[dict[str, str]],
    max_tokens: int = THAILLM_MAX_TOKENS,
    temperature: float = THAILLM_TEMPERATURE,
) -> dict[str, Any]:
    payload = {
        "model": THAILLM_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    body = json.dumps(payload, ensure_ascii=False)
    last_error: Exception | None = None
    for attempt in range(1, REQUEST_MAX_RETRIES + 1):
        try:
            result = subprocess.run(
                [
                    "curl",
                    "-sS",
                    "--fail-with-body",
                    THAILLM_URL,
                    "-H",
                    "Content-Type: application/json",
                    "-H",
                    f"apikey: {thaillm_api_key}",
                    "-d",
                    body,
                ],
                capture_output=True,
                text=True,
                check=True,
            )
            raw = result.stdout.strip()
            if not raw:
                raise RuntimeError(
                    f"ThaiLLM returned empty stdout. stderr={result.stderr!r}"
                )
            try:
                data = json.loads(raw)
            except json.JSONDecodeError as exc:
                preview = raw[:1000]
                raise RuntimeError(
                    f"ThaiLLM returned non-JSON response: {preview}"
                ) from exc
            choices = data.get("choices", [])
            if not choices:
                raise RuntimeError("ThaiLLM response contained no choices")
            message = choices[0].get("message", {})
            content = message.get("content", "")
            finish_reason = choices[0].get("finish_reason")
            return {
                "raw_content": content,
                "finish_reason": finish_reason,
            }
        except (
            subprocess.CalledProcessError,
            json.JSONDecodeError,
            RuntimeError,
            ValueError,
        ) as exc:
            stderr = (
                exc.stderr if isinstance(exc, subprocess.CalledProcessError) else ""
            )
            stdout = (
                exc.stdout if isinstance(exc, subprocess.CalledProcessError) else ""
            )
            if isinstance(exc, subprocess.CalledProcessError):
                last_error = RuntimeError(
                    f"curl failed with code {exc.returncode}: {stderr or stdout}"
                )
            else:
                last_error = exc
            if attempt == REQUEST_MAX_RETRIES:
                break
            time.sleep(REQUEST_SLEEP_SECONDS * attempt)
    raise RuntimeError(
        f"ThaiLLM call failed after {REQUEST_MAX_RETRIES} attempts: {last_error}"
    )


SYSTEM_MESSAGE = """คุณเป็นผู้เชี่ยวชาญสินค้าอิเล็กทรอนิกส์ของร้านฟ้าใหม่ คุณตอบคำถามอย่างแม่นยำโดยอิงหลักฐานที่ให้มาเท่านั้น คุณเก่งเรื่องการคำนวณราคา คะแนนสะสม และการเปรียบเทียบสเปคสินค้า""".strip()


def build_user_prompt(
    question_row: pd.Series,
    retrieved_chunks: list[dict[str, Any]],
    budget_context: str | None = None,
) -> str:
    choices_text = "\n".join(
        f"{answer_id}. {question_row[f'choice_{answer_id}']}"
        for answer_id in range(1, 11)
    )
    evidence_text = "\n\n".join(
        f"[{idx + 1}] id={chunk['id']} | path={chunk['file_path']} | heading={' > '.join(chunk['heading_path'])}\n{chunk['chunk_text']}"
        for idx, chunk in enumerate(retrieved_chunks)
    )
    if budget_context:
        evidence_text = f"{budget_context}\n\n---\n\n{evidence_text}"
    instructions = """
คุณเป็นผู้ช่วยตอบคำถามหลายตัวเลือกสำหรับร้านฟ้าใหม่ (ร้านขายสินค้าอิเล็กทรอนิกส์ของไทย)
ใช้หลักฐานที่ให้มาเท่านั้น

ขั้นตอน:
1. อ่านหลักฐานอย่างละเอียด
2. วิเคราะห์ใน <think>...</think> ว่าหลักฐานตอบคำถามอย่างไร
   - ถ้าต้องคำนวณ (รวมราคา, คำนวณ Points, นับจำนวนสินค้า) ให้แสดงขั้นตอนการคำนวณ
   - ถ้าต้องเปรียบเทียบสินค้า ให้ระบุความแตกต่างที่พบ
3. ตอบเลขคำตอบเพียงตัวเดียวหลัง </think>

กฎการเลือกคำตอบ:
- เลือก 1-8: เมื่อหลักฐานสนับสนุนตัวเลือกใดตัวเลือกหนึ่ง แม้จะสนับสนุนบางส่วนก็ให้เลือกตัวเลือกที่ใกล้เคียงที่สุด
- เลือก 9: เฉพาะเมื่อคำถามเกี่ยวกับร้านฟ้าใหม่ แต่ค้นหลักฐานทั้งหมดแล้วไม่มีข้อมูลที่เกี่ยวข้องเลย ห้ามเลือก 9 ถ้ามีข้อมูลบางส่วนที่ช่วยตอบได้
- เลือก 10: เฉพาะเมื่อคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่ สินค้า หรือนโยบายร้านเลย (เช่น ถามเรื่องสูตรอาหาร การเมือง ท่องเที่ยว ดอกเบี้ย ตั๋วเครื่องบิน วันหยุดราชการ)

รูปแบบคำตอบ:
<think>วิเคราะห์ที่นี่</think>
[ตัวเลขคำตอบ 1-10]
""".strip()
    return f"{instructions}\n\nคำถาม:\n{question_row['question']}\n\nตัวเลือก:\n{choices_text}\n\nหลักฐานที่ดึงมา:\n{evidence_text}\n\nวิเคราะห์ใน <think> แล้วตอบเลขเดียว:"


In [9]:
def support_bonus(choice_text: str, chunk_text: str) -> float:
    choice_tokens = extract_support_tokens(choice_text)
    chunk_tokens = extract_support_tokens(chunk_text)
    if not choice_tokens or not chunk_tokens:
        return 0.0
    overlap = choice_tokens & chunk_tokens
    if not overlap:
        return 0.0
    overlap_ratio = len(overlap) / max(len(choice_tokens), 1)
    numeric_overlap = sum(1 for token in overlap if NUMBER_TOKEN_RE.fullmatch(token))
    table_bonus = 0.15 if "|" in chunk_text else 0.0
    return overlap_ratio * 0.8 + numeric_overlap * 0.2 + table_bonus


def looks_store_related(
    question_text: str, retrieved_chunks: list[dict[str, Any]]
) -> bool:
    normalized = normalize_text(question_text)
    if any(term in normalized for term in DOMAIN_TERMS if len(term) >= 2):
        return True
    return any(chunk["score"] > QUESTION_WEAK_SCORE for chunk in retrieved_chunks)


def looks_out_of_domain(
    question_text: str, retrieved_chunks: list[dict[str, Any]]
) -> bool:
    normalized = normalize_text(question_text)
    weak_retrieval = (
        max((chunk["score"] for chunk in retrieved_chunks), default=0.0)
        < QUESTION_WEAK_SCORE
    )
    has_hint = any(term in normalized for term in OUT_OF_DOMAIN_HINTS)
    long_story = len(question_text) >= OUT_OF_DOMAIN_LENGTH
    has_store_term = any(term in normalized for term in DOMAIN_TERMS if len(term) >= 2)
    return weak_retrieval and (has_hint or long_story) and not has_store_term


def fallback_answer(
    question_row: pd.Series, retrieved_chunks: list[dict[str, Any]]
) -> tuple[int, str]:
    question_text = str(question_row["question"])
    if looks_out_of_domain(question_text, retrieved_chunks):
        return 10, "fallback_out_of_domain"

    choice_rows = []
    for answer_id in range(1, 9):
        choice_text = str(question_row[f"choice_{answer_id}"])
        scores = [
            chunk["score"] + support_bonus(choice_text, chunk["chunk_text"])
            for chunk in retrieved_chunks
        ]
        choice_rows.append(
            {
                "answer_id": answer_id,
                "choice_text": choice_text,
                "score": max(scores) if scores else 0.0,
            }
        )
    choice_scores_df = (
        pd.DataFrame(choice_rows)
        .sort_values(["score", "answer_id"], ascending=[False, True])
        .reset_index(drop=True)
    )
    best_choice = choice_scores_df.iloc[0]
    best_score = float(best_choice["score"])
    if best_score < QUESTION_WEAK_SCORE and looks_store_related(
        question_text, retrieved_chunks
    ):
        return 9, "fallback_missing_info"
    return int(best_choice["answer_id"]), "fallback_best_choice"


def validate_thaillm_answer(payload: dict[str, Any]) -> int:
    answer_id = payload.get("answer_id")
    if isinstance(answer_id, str) and answer_id.isdigit():
        answer_id = int(answer_id)
    if not isinstance(answer_id, int) or not 1 <= answer_id <= 10:
        raw_content = payload.get("raw_content", "")
        answer_id = extract_answer_id_from_text(raw_content)
    return answer_id


RETRY_PROMPT_SUFFIX = """หลักฐานด้านบนมีข้อมูลที่เกี่ยวข้องกับคำถาม กรุณาอ่านหลักฐานอย่างละเอียดอีกครั้ง
คุณต้องเลือกตัวเลือก 1-8 ที่ตรงกับหลักฐานมากที่สุด
เลือก 9 ได้เฉพาะเมื่อมั่นใจ 100% ว่าไม่มีข้อมูลในหลักฐานเลย

<think>วิเคราะห์อีกครั้ง</think>
ตอบเลขเดียว:"""


def answer_question(question_row: pd.Series) -> dict[str, Any]:
    question_id = int(question_row["id"])
    question_text = str(question_row["question"])
    retrieved_chunks = retrieve_chunks(question_text, top_k=TOP_K)
    retrieved_paths = [chunk["file_path"] for chunk in retrieved_chunks]
    top_score = max((chunk["score"] for chunk in retrieved_chunks), default=0.0)
    budget_context = get_budget_context(question_text)
    user_prompt = build_user_prompt(question_row, retrieved_chunks, budget_context)
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_prompt},
    ]
    used_fallback = False
    fallback_reason = None
    raw_model_output = None
    reason = None
    evidence_sources = []
    retried = False
    payload = call_thaillm(messages)
    raw_model_output = payload.get("raw_content")
    try:
        answer_id = validate_thaillm_answer(payload)
        reason = payload.get("reason")
        evidence_sources = payload.get("evidence_sources", [])
        # Two-pass retry: if answer is 9 or 10 but retrieval is strong and question is store-related
        if (
            answer_id in (9, 10)
            and top_score >= RETRY_ON_9_10_THRESHOLD
            and looks_store_related(question_text, retrieved_chunks)
        ):
            retry_messages = messages + [
                {"role": "assistant", "content": raw_model_output},
                {"role": "user", "content": RETRY_PROMPT_SUFFIX},
            ]
            retry_payload = call_thaillm(retry_messages)
            retry_raw = retry_payload.get("raw_content", "")
            try:
                retry_answer = extract_answer_id_from_text(retry_raw)
                if 1 <= retry_answer <= 8:
                    answer_id = retry_answer
                    raw_model_output = f"{raw_model_output}\n[RETRY]: {retry_raw}"
                    retried = True
                elif retry_answer != answer_id:
                    answer_id = retry_answer
                    raw_model_output = f"{raw_model_output}\n[RETRY]: {retry_raw}"
                    retried = True
            except ValueError:
                pass  # Keep original answer
    except Exception as exc:
        answer_id, fallback_reason = fallback_answer(question_row, retrieved_chunks)
        used_fallback = True
        raw_model_output = f"INVALID_MODEL_OUTPUT: {raw_model_output}\nERROR: {exc}"
        reason = fallback_reason
        evidence_sources = [chunk["id"] for chunk in retrieved_chunks[:2]]
    time.sleep(REQUEST_SLEEP_SECONDS)
    return {
        "id": question_id,
        "question": question_text,
        "predicted_answer": int(answer_id),
        "raw_model_output": raw_model_output,
        "reason": reason,
        "evidence_sources": evidence_sources,
        "retrieved_paths": retrieved_paths,
        "top_score": float(top_score),
        "used_fallback": used_fallback,
        "fallback_reason": fallback_reason,
        "retried": retried,
    }


In [10]:
results = []
for _, question_row in tqdm(
    questions_df.iterrows(), total=len(questions_df), desc="Answer questions"
):
    result = answer_question(question_row)
    results.append(result)
    print(
        f"Question {result['id']}: answer={result['predicted_answer']} fallback={result['used_fallback']} retried={result['retried']}"
    )

debug_df = pd.DataFrame(results)
debug_df["raw_model_output"] = debug_df["raw_model_output"].str.replace(
    "\n", "\\n", regex=False
)
submission_df = sample_submission_df.copy()
if MAX_QUESTIONS is not None:
    submission_df = submission_df.head(MAX_QUESTIONS).copy()
submission_df["answer"] = debug_df["predicted_answer"].tolist()

assert len(submission_df) == len(debug_df), "Submission length mismatch"
assert submission_df["id"].tolist() == debug_df["id"].tolist(), (
    "Submission ids do not match question ids"
)
assert submission_df["answer"].between(1, 10).all(), "Invalid answer outside 1-10"

OUTPUT_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
debug_df.to_csv(OUTPUT_DEBUG_PATH, index=False)

display(submission_df.head())
display(
    debug_df[
        [
            "id",
            "predicted_answer",
            "top_score",
            "used_fallback",
            "fallback_reason",
            "retried",
            "retrieved_paths",
        ]
    ].head(10)
)
print(f"Saved submission to: {OUTPUT_SUBMISSION_PATH}")
print(f"Saved debug table to: {OUTPUT_DEBUG_PATH}")
print(f"\nAnswer distribution:")
print(debug_df["predicted_answer"].value_counts().sort_index())
print(f"\nRetried: {debug_df['retried'].sum()} questions")
print(f"Fallbacks: {debug_df['used_fallback'].sum()} questions")


Answer questions:   0%|          | 0/100 [00:00<?, ?it/s]

Question 1: answer=5 fallback=False retried=True
Question 2: answer=7 fallback=False retried=False
Question 3: answer=2 fallback=False retried=False
Question 4: answer=9 fallback=False retried=False
Question 5: answer=6 fallback=False retried=False


RuntimeError: ThaiLLM call failed after 3 attempts: curl failed with code 22: curl: (22) The requested URL returned error: 504


In [ ]:
print("Question 1 smoke test")

question_row = questions_df.iloc[0]
retrieved_chunks = retrieve_chunks(str(question_row["question"]), top_k=5)

smoke_test_chunks = []
for chunk in retrieved_chunks:
    smoke_test_chunks.append(
        {
            **chunk,
            "chunk_text": chunk["chunk_text"][:500],
        }
    )

prompt = build_user_prompt(question_row, smoke_test_chunks)
payload = {
    "model": THAILLM_MODEL,
    "messages": [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ],
    "max_tokens": THAILLM_MAX_TOKENS,
    "temperature": THAILLM_TEMPERATURE,
}
body = json.dumps(payload, ensure_ascii=False)

print("QUESTION:")
print(question_row["question"])
print("\nTOP CHUNKS:")
for idx, chunk in enumerate(smoke_test_chunks, start=1):
    heading = " > ".join(chunk["heading_path"])
    print(f"[{idx}] {chunk['id']} | score={chunk['score']:.4f} | {heading}")

result = subprocess.run(
    [
        "curl",
        "-sS",
        "--fail-with-body",
        THAILLM_URL,
        "-H",
        "Content-Type: application/json",
        "-H",
        f"apikey: {thaillm_api_key}",
        "-d",
        body,
    ],
    capture_output=True,
    text=True,
    check=True,
)

raw = result.stdout.strip()
if not raw:
    raise RuntimeError(
        f"Question 1 smoke test returned empty stdout. stderr={result.stderr!r}"
    )

print("\nRAW RESPONSE:")
print(raw)

data = json.loads(raw)
content = data["choices"][0]["message"].get("content", "")

print("\nMODEL CONTENT:")
print(content)
print("\nFINISH REASON:")
print(data["choices"][0].get("finish_reason"))

try:
    answer_id = extract_answer_id_from_text(content)
    print(f"\nEXTRACTED ANSWER: {answer_id}")
except Exception as e:
    print(f"\nFailed to extract answer: {e}")
